# Analyzing DHS microdata for Nigeria

Nigeria 2018 - J:/DATA/DHS_PROG_DHS/NGA/2018/

## Documentation

DHS 7 recode manual for variable definitions: https://www.dhsprogram.com/pubs/pdf/DHSG4/Recode7_DHS_10Sep2018_DHSG4.pdf.
However, it does not state which variables are in which file.
The `.MAP` files alongside each data file list the variables in it and what they mean.

Hemoglobin variables of interest: 
- HA53: Hemoglobin level in g/dl with 1 implied dcimal
- HA54: Currently pregnant
- HA55 Result of Hemoglobin measuring.
- HA56 Hemoglobin level adjusted by altitude in g/dl with 1 implied decimal. 

Wealth index variables of interest: 
- HV270: The wealth index is a composite measure of a household's cumulative living standard.
The wealth index is calculated using easy-to-collect data on a household’s ownership of
selected assets, such as televisions and bicycles; materials used for housing construction; and
types of water access and sanitation facilities.
Generated with a statistical procedure known as principal components analysis, the wealth
index places individual households on a continuous scale of relative wealth. DHS separates
all interviewed households into five wealth quintiles to compare the influence of wealth on
various population, health and nutrition indicators. The wealth index is presented in the DHS
Final Reports and survey datasets as a background characteristic
- HV271: Wealth index factor score (5 decimals) 

Pregnancy variables of interest: 
- HML18: Pregnancy status from individual questionnaire. For complete woman’s interviews this is
taken from V213. For incomplete woman's interview with anemia testing the pregnancy
status is taken from this section.
BASE: Women with a completed individual questionnaire or when available information
from the anemia testing section.

List of datasets: https://www.dhsprogram.com/data/dataset/Nigeria_Standard-DHS_2018.cfm?flag=1

Instructions on how to calculate everything can be found at: https://www.dhsprogram.com/pubs/pdf/DHSG1/Guide_to_DHS_Statistics_DHS-7_v2.pdf

In [ ]:
import pandas as pd, numpy as np

%load_ext autoreload
%autoreload 2

!date

## Load data, name columns

In [ ]:
directory = "/snfs1/DATA/DHS_PROG_DHS/NGA/2018/"

### WRA

In [ ]:
%%time

raw_wra_data = pd.read_stata(directory + "NGA_DHS7_2018_WN_NGIR7AFL_Y2019M11D05.DTA")

In [ ]:
wra_data = raw_wra_data.copy()
wra_data

In [ ]:
wra_columns = {
    "v001": "cluster_number",
    "v002": "household_number",
    "v003": "line_number",
    "v005": "weight",
    "v008": "interview_date",
    "v011": "date_of_birth",
    "v190": "wealth_quintile",
}
wra_data = wra_data[wra_columns.keys()].rename(columns=wra_columns)

In [ ]:
def recode_wealth_quintile(df):
    return df.map(
        {
            "poorest": "lowest",
            "poorer": "second",
            "middle": "middle",
            "richer": "fourth",
            "richest": "highest",
        }
    )

In [ ]:
wra_data["wealth_quintile"] = recode_wealth_quintile(wra_data.wealth_quintile)

In [ ]:
wra_data["weight"] = wra_data.weight / 1_000_000

### Births

In [ ]:
birth_columns = {
    "v005": "weight",
    "v008": "interview_date",
    "v190": "wealth_quintile",
    "b3": "birth_date",
    "b20": "duration_of_pregnancy",
    "m18": "size_of_child",
    "m19": "birth_weight_kilograms",
}

birth_data = pd.read_stata(
    directory + "NGA_DHS7_2018_BR_NGBR7AFL_Y2019M11D05.DTA",
    columns=birth_columns.keys(),
)
birth_data

In [ ]:
birth_data = birth_data[birth_columns.keys()].rename(columns=birth_columns)
birth_data["wealth_quintile"] = recode_wealth_quintile(birth_data.wealth_quintile)
birth_data["weight"] = birth_data.weight / 1_000_000
birth_data

### Household members

In [ ]:
%%time

hhm_data = pd.read_stata(directory + "NGA_DHS7_2018_HHM_NGPR7AFL_Y2019M11D05.DTA")
hhm_data

In [ ]:
hhm_columns = {
    "hv001": "cluster_number",
    "hv002": "household_number",
    "hvidx": "line_number",
    "hml18": "currently_pregnant",
    "ha1": "age",
    "hv270": "wealth_quintile",
    "ha53": "hemoglobin_raw",
    "ha56": "hemoglobin_adjusted",
    "ha57": "anemia",
}
hhm_data = hhm_data[hhm_columns.keys()].rename(columns=hhm_columns)

In [ ]:
hhm_data["wealth_quintile"] = recode_wealth_quintile(hhm_data.wealth_quintile)

In [ ]:
for col in ["hemoglobin_raw", "hemoglobin_adjusted"]:
    hhm_data[col] = (
        hhm_data[col]
        .astype(str)
        .replace({"not present": np.nan, "refused": np.nan, "other": np.nan})
        .astype(float)
    )

### Siblings

Sibling survival data appears to only be available as a kind of side table-within-a-table on WRA.

It is labeled "MM" because it is used to calculate maternal mortality (among other things).

In [ ]:
respondent_column_names = {
    "v008": "interview_date",
    "v190": "wealth_quintile",
    "v005": "weight",
}

# These are suffixed with an underscore and an integer, e.g. mm1_01
sibling_column_names = {
    # MM1                    Sex of sibling                                  7156    1    N    I   20    0   No   No
    "mm1": "sex",
    # MM2                    Survival status of sibling                      7176    1    N    I   20    0   No   No
    "mm2": "survival_status",
    # MM3                    Sibling's current age                           7196    2    N    I   20    0   No   No
    "mm3": "current_age",
    # MM4                    Sibling's date of birth (CMC)                   7236    4    N    I   20    0   No   No
    "mm4": "date_of_birth",
    # MM8                    Date of death of sibling (CMC)                  7416    4    N    I   20    0   No   No
    "mm8": "date_of_death",
    # MM7                    Sibling's age at death                          7376    2    N    I   20    0   No   No
    "mm7": "age_at_death",
    # MM9                    Sibling's death and pregnancy                   7496    2    N    I   20    0   No   No
    "mm9": "pregnancy_category",
    # MM16                   Sibling's death due to violence or accident     7816    1    N    I   20    0   No   No
    "mm16": "death_violence_or_accident",
}

In [ ]:
sibling_data = raw_wra_data[
    [
        c
        for c in raw_wra_data.columns
        if c in respondent_column_names.keys()
        or c.split("_")[0] in sibling_column_names.keys()
    ]
].copy()
sibling_data

In [ ]:
# inspired by https://stackoverflow.com/a/67393747/
sibling_data_reshaped = sibling_data[
    [c for c in sibling_data.columns if c.split("_")[0] in sibling_column_names.keys()]
].copy()
sibling_data_reshaped.columns = sibling_data_reshaped.columns.str.split(
    "_", expand=True
)
sibling_data_reshaped

In [ ]:
sibling_data_reshaped[list(respondent_column_names.keys())] = sibling_data[
    list(respondent_column_names.keys())
]
sibling_data_reshaped

In [ ]:
# Get a row per sibling
sibling_data_reshaped = (
    sibling_data_reshaped.set_index(list(respondent_column_names.keys()))
    .swaplevel(axis=1)
    .stack(0)
    .reset_index()
    .drop(columns=[f"level_{len(respondent_column_names)}"])
)
sibling_data_reshaped

In [ ]:
sibling_data = (
    sibling_data_reshaped[
        list(respondent_column_names.keys()) + list(sibling_column_names.keys())
    ]
    .rename(columns=respondent_column_names)
    .rename(columns=sibling_column_names)
)
sibling_data["wealth_quintile"] = recode_wealth_quintile(sibling_data.wealth_quintile)
sibling_data["weight"] = sibling_data.weight / 1_000_000
sibling_data

In [ ]:
# "A total of 219,561 siblings were recorded..." (p. 372)
len(sibling_data)

## WRA

### Hemoglobin among pregnancies

In [ ]:
id_columns = ["cluster_number", "household_number", "line_number"]
other_overlapping_columns = (set(wra_data.columns) & set(hhm_data.columns)) - set(
    id_columns
)
other_overlapping_columns

In [ ]:
wra_hhm_joined = wra_data.merge(
    hhm_data,
    on=id_columns,
    suffixes=("_wra", "_hhm"),
    how="left",
)
wra_hhm_joined

In [ ]:
for col in other_overlapping_columns:
    assert (wra_hhm_joined[f"{col}_wra"] == wra_hhm_joined[f"{col}_hhm"]).all()
    wra_hhm_joined[col] = wra_hhm_joined[f"{col}_wra"]
    wra_hhm_joined = wra_hhm_joined.drop(columns=[f"{col}_wra", f"{col}_hhm"])

In [ ]:
wra_hhm_joined.currently_pregnant.value_counts(dropna=False)

In [ ]:
pregnant_data = wra_hhm_joined[wra_hhm_joined.currently_pregnant == "pregnant"].copy()
pregnant_data

In [ ]:
# https://stackoverflow.com/a/2415343/ with some tweaks
def weighted_avg_and_std(values, weights):
    """
    Return the weighted average and standard deviation.

    They weights are in effect first normalized so that they
    sum to 1 (and so they must not all be 0).

    values, weights -- NumPy ndarrays with the same shape.
    """
    is_nan = np.isnan(values)
    values = values[~is_nan]
    weights = weights[~is_nan]
    average = np.average(values, weights=weights)
    # Fast and numerically precise:
    variance = np.average((values - average) ** 2, weights=weights)
    return pd.Series(
        {
            "mean": average,
            "sd": np.sqrt(variance),
            # https://ngreifer.github.io/WeightIt/reference/ESS.html
            "effective_sample_size": (weights.sum() ** 2) / (weights**2).sum(),
        }
    )

In [ ]:
# NOTE: Different from table 11.13 in report, where this is 1,542
# I tried a couple different things but couldn't figure it out
pregnant_data.anemia.notnull().sum()

In [ ]:
# Within rounding error of table 11.13 value
weighted_avg_and_std(
    pregnant_data[pregnant_data.anemia.notnull()].anemia == "severe",
    pregnant_data[pregnant_data.anemia.notnull()].weight,
)

In [ ]:
# Within rounding error of table 11.13 value for any anemia
weighted_avg_and_std(
    pregnant_data[pregnant_data.anemia.notnull()].anemia.isin(
        ["severe", "moderate", "mild"]
    ),
    pregnant_data[pregnant_data.anemia.notnull()].weight,
)

In [ ]:
assert (
    (pregnant_data[pregnant_data.anemia.notnull()].anemia == "severe")
    == (pregnant_data[pregnant_data.anemia.notnull()].hemoglobin_adjusted < 70)
).all()

In [ ]:
assert (
    (
        pregnant_data[pregnant_data.anemia.notnull()].anemia.isin(
            ["severe", "moderate", "mild"]
        )
    )
    == (pregnant_data[pregnant_data.anemia.notnull()].hemoglobin_adjusted < 110)
).all()

In [ ]:
# Very wide age bins -- but still not enough sample size
age_bin_edges = [15, 25, 30, 50]
age_bin_edges

In [ ]:
pregnant_data["age_group"] = pd.IntervalIndex(
    pd.cut(pregnant_data.age, age_bin_edges, right=False)
)

In [ ]:
# NOTE: We don't use this age stratification because it shows inconsistent patterns and fluctuations
(
    pregnant_data.groupby(["age_group", "wealth_quintile"])
    .apply(lambda df: weighted_avg_and_std(df.hemoglobin_adjusted, weights=df.weight))
    .sort_index()
)

In [ ]:
hemoglobin_disparities = (
    pregnant_data.groupby(["wealth_quintile"])
    .apply(lambda df: weighted_avg_and_std(df.hemoglobin_adjusted, weights=df.weight))
    .sort_index()
)
hemoglobin_disparities

In [ ]:
hemoglobin_disparities = hemoglobin_disparities.reset_index()
hemoglobin_disparities["sex"] = "Female"
hemoglobin_disparities = hemoglobin_disparities.set_index(["sex", "wealth_quintile"])
hemoglobin_disparities

In [ ]:
pregnancy_sim_input_data_dir = (
    "../../0200_pregnancy_sim/src/vivarium_gates_lsff_by_wealth_quintile/data/raw_data"
)

In [ ]:
hemoglobin_disparities["mean"].rename("value").to_csv(
    f"{pregnancy_sim_input_data_dir}/hemoglobin/mean_disparities/nigeria.csv"
)

In [ ]:
hemoglobin_disparities["sd"].rename("value").to_csv(
    f"{pregnancy_sim_input_data_dir}/hemoglobin/sd_disparities/nigeria.csv"
)

### Wealth quintile probabilities

Intuitively, you might think these would be equal; but we are looking at a subpopulation (pregnancies) that skews poorer.

In [ ]:
assert pregnant_data.wealth_quintile.notnull().all()

In [ ]:
wealth_quintile_probabilities = []

for quintile in pregnant_data.wealth_quintile.unique():
    quintile_info = pd.DataFrame(
        weighted_avg_and_std(
            pregnant_data.wealth_quintile == quintile, pregnant_data.weight
        )
    ).T
    quintile_info.insert(0, "wealth_quintile", quintile)
    wealth_quintile_probabilities.append(quintile_info)

wealth_quintile_probabilities = pd.concat(
    wealth_quintile_probabilities, ignore_index=True
)
wealth_quintile_probabilities.sort_values("mean")

In [ ]:
wealth_quintile_probabilities["mean"].sum()

In [ ]:
wealth_quintile_probabilities = (
    wealth_quintile_probabilities.set_index("wealth_quintile")["mean"]
    .to_frame()
    .T.reset_index(drop=True)
)
wealth_quintile_probabilities.columns.name = None
wealth_quintile_probabilities

In [ ]:
wealth_quintile_probabilities.insert(0, "sex", "Female")
wealth_quintile_probabilities

In [ ]:
wealth_quintile_probabilities.to_csv(
    f"{pregnancy_sim_input_data_dir}/wealth_quintile_probabilities/nigeria.csv",
    index=False,
)

### Maternal mortality ratio

#### Maternal mortality rate

In [ ]:
sibling_data.survival_status.value_counts()

In [ ]:
sibling_data.pregnancy_category.value_counts()

In [ ]:
sibling_data.death_violence_or_accident.value_counts()

In [ ]:
# https://dhsprogram.com/Data/Guide-to-DHS-Statistics/Adult_Mortality_Rates.htm#Calculation1
sibling_data["exposure_start"] = np.maximum(
    sibling_data.date_of_birth + 12 * 15, sibling_data.interview_date - 84
)  # aka lowlim
# aka upplim
sibling_data["exposure_end"] = np.minimum(
    np.where(
        sibling_data.survival_status == "alive",
        sibling_data.interview_date - 1,
        sibling_data.date_of_death,
    ),
    sibling_data.date_of_birth + 12 * 50 - 1,
)
sibling_data["exposure"] = (
    (sibling_data.exposure_end - sibling_data.exposure_start) + 1
).clip(lower=0)

In [ ]:
sibling_data.exposure.value_counts()

In [ ]:
sibling_data["adult_death"] = (
    (sibling_data.survival_status == "dead")
    & (sibling_data.date_of_death - sibling_data.date_of_birth >= 15.0 * 12)
    & (sibling_data.date_of_death - sibling_data.date_of_birth < 50.0 * 12)
    & (sibling_data.exposure > 0)
    & (sibling_data.date_of_death >= sibling_data.exposure_start)
    & (sibling_data.date_of_death <= sibling_data.exposure_end)
)

In [ ]:
# Matches table 14.2
(
    sibling_data[sibling_data.date_of_birth.notnull()]
    .assign(weighted_exposure=lambda df: df.exposure * df.weight)
    .groupby("sex")
    .weighted_exposure.sum()
    / 12
)

In [ ]:
# Matches table 14.2
sibling_data.assign(
    weighted_adult_dealth=lambda df: df.adult_death * df.weight
).groupby("sex").weighted_adult_dealth.sum()

In [ ]:
sibling_data.death_violence_or_accident.value_counts()

In [ ]:
sibling_data.pregnancy_category.value_counts()

In [ ]:
female_siblings = sibling_data[sibling_data.sex == "female"].copy()
female_siblings["maternal_death"] = (
    (female_siblings.adult_death)
    & (
        female_siblings.pregnancy_category.isin(
            ["died during delivery", "died while pregnant", "6 weeks after delivery"]
        )
    )
    & (~female_siblings.death_violence_or_accident.isin(["violence", "accident"]))
)

In [ ]:
# Table 14.4 reports 451 maternal deaths
(female_siblings.maternal_death * female_siblings.weight).sum()

In [ ]:
# Table 14.4 reports 480,382
(female_siblings.exposure * female_siblings.weight / 12).sum()

In [ ]:
def maternal_mortality_rate(df):
    return ((df.maternal_death * df.weight).sum() * 1_000) / (
        (df.exposure * df.weight) / 12
    ).sum()

In [ ]:
# "the maternal mortality rate among women age 15-49 is 0.92 deaths per 1,000 woman-years of exposure." (p. 374)
# TODO: These are not age-standardized! We figure the *disparity* probably isn't way off.
# Should standardize according to the approach from https://github.com/LateraOlana/Fertility_SIM_DHS/blob/main/fertility/Latera_Zebb_Coworking.ipynb
maternal_mortality_rate(female_siblings)

In [ ]:
maternal_mortality_rates = female_siblings.groupby("wealth_quintile").apply(
    maternal_mortality_rate
)
maternal_mortality_rates

#### General fertility rate

In [ ]:
fertility_event_data = birth_data.copy()
fertility_event_data["birth_in_period"] = (
    (fertility_event_data.interview_date - fertility_event_data.birth_date) >= 1
) & ((fertility_event_data.interview_date - fertility_event_data.birth_date) <= 36)
fertility_event_data["weighted_birth_in_period"] = (
    fertility_event_data.birth_in_period * fertility_event_data.weight
)

In [ ]:
fertility_event_data.weighted_birth_in_period.sum()

In [ ]:
fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()

In [ ]:
fertility_exposure_data = wra_data.copy()
fertility_exposure_data["exposure_start"] = np.maximum(
    fertility_exposure_data.date_of_birth + 12 * 15,
    fertility_exposure_data.interview_date - 36,
)  # aka lowlim
# aka upplim
fertility_exposure_data["exposure_end"] = np.minimum(
    fertility_exposure_data.interview_date - 1,
    fertility_exposure_data.date_of_birth + 12 * 45,
)
fertility_exposure_data["exposure"] = (
    (fertility_exposure_data.exposure_end - fertility_exposure_data.exposure_start) + 1
).clip(lower=0)
fertility_exposure_data["weighted_exposure"] = (
    fertility_exposure_data.exposure * fertility_exposure_data.weight
)

In [ ]:
# Within rounding error of value reported in Table 5.1
fertility_event_data.weighted_birth_in_period.sum() * 1_000 / (
    fertility_exposure_data.weighted_exposure.sum() / 12
)

In [ ]:
gfr_by_wealth = (
    fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()
    * 1_000
    / (fertility_exposure_data.groupby("wealth_quintile").weighted_exposure.sum() / 12)
)
gfr_by_wealth

In [ ]:
maternal_disorders_incidence_disparities = (
    (maternal_mortality_rates / gfr_by_wealth)
    .rename("value")
    .rename_axis("wealth_quintile")
    .reset_index()
)
maternal_disorders_incidence_disparities.insert(0, "sex", "Female")
maternal_disorders_incidence_disparities

In [ ]:
maternal_disorders_incidence_disparities.to_csv(
    f"{pregnancy_sim_input_data_dir}/maternal_disorders_incidence_disparities/nigeria.csv",
    index=False,
)

## LBWSG

### Birth weight

In [ ]:
birth_data["birth_weight_kilograms"] = birth_data.birth_weight_kilograms.replace(
    {"not weighed at birth": np.nan, "don't know": np.nan}
).astype(float)

In [ ]:
weighted_avg_and_std(birth_data.birth_weight_kilograms, birth_data.weight)

In [ ]:
birth_weight_disparities = birth_data.groupby("wealth_quintile").apply(
    lambda df: weighted_avg_and_std(df.birth_weight_kilograms, df.weight)
)
birth_weight_disparities

In [ ]:
birth_weight_disparities = (
    birth_weight_disparities["mean"].rename("value").reset_index()
)
birth_weight_disparities

In [ ]:
child_sim_input_data_dir = "../../0300_child_sim/src/vivarium_gates_lsff_by_wealth_quintile_child/data/raw_data"

birth_weight_disparities.to_csv(
    f"{child_sim_input_data_dir}/birth_weight_disparities/nigeria.csv", index=False
)

### Short gestation

All we have here is a self-reported duration of pregnancy.

In [ ]:
# Basically no difference in mean
birth_data.groupby("wealth_quintile").apply(
    lambda df: weighted_avg_and_std(df.duration_of_pregnancy, df.weight)
)

In [ ]:
birth_data["short_gestation"] = np.where(
    birth_data.duration_of_pregnancy.isnull(),
    np.nan,
    birth_data.duration_of_pregnancy < 9.0,
)

In [ ]:
weighted_avg_and_std(birth_data.short_gestation, birth_data.weight)

In [ ]:
birth_data.groupby("wealth_quintile").apply(
    lambda df: weighted_avg_and_std(df.short_gestation, df.weight)
)

The trends in short gestation don't make intuitive sense (?), so we do not plan to use them in the sim.